# 07. Multi-Dataset Evaluation (FD001 – FD004)

This notebook extends the existing FD001-only pipeline to **all four C-MAPSS datasets**. For each dataset we:
1. Load raw data and engineer features (rolling mean/std, normalization)
2. Train **XGBoost**, **SVR**, and **LSTM** models
3. Evaluate on the test set using **RMSE**, **MAE**, and **NASA Score**

Results are collected into a unified cross-dataset comparison table.

| Dataset | Train Engines | Test Engines | Operating Conditions | Fault Modes |
|---------|--------------|-------------|---------------------|-------------|
| FD001 | 100 | 100 | 1 | 1 |
| FD002 | 260 | 259 | 6 | 1 |
| FD003 | 100 | 100 | 1 | 2 |
| FD004 | 249 | 248 | 6 | 2 |

**Runtime:** ~15-25 min on Colab GPU (T4).

---
## 1. Setup & Imports

In [ ]:
# ── Google Drive Mount ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/nasa_turbofan_project'
DATA = f'{BASE}/data'

# ── Standard Imports ────────────────────────────────────────────
import os, sys, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.svm import SVR

from xgboost import XGBRegressor

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ── NASA Score (inline to avoid file-copy hassles) ──────────────
def nasa_score(y_true, y_pred):
    d = y_pred - y_true
    return np.sum(np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1))

# ── Device ──────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'\nBase path: {BASE}')
print('Setup complete!')

---
## 2. Configuration

In [ ]:
# ── Datasets to evaluate ────────────────────────────────────────
DATASETS = ['FD001', 'FD002', 'FD003', 'FD004']

# ── Column names (same for all 4 datasets) ─────────────────────
COLS = ['unit', 'cycle', 'op1', 'op2', 'op3'] + [f's{i}' for i in range(1, 22)]

# ── Pipeline parameters ─────────────────────────────────────────
RUL_CAP          = 125    # max RUL label value
ROLLING_WINDOW   = 30     # for rolling mean/std features
SEQUENCE_LENGTH  = 30     # LSTM sliding window length
VARIANCE_THRESH  = 0.001  # sensors with std below this are dropped
SVR_SAMPLE_SIZE  = 5000   # max samples for SVR training

# ── LSTM hyperparameters ────────────────────────────────────────
LSTM_HIDDEN  = 100
LSTM_LAYERS  = 2
LSTM_DROPOUT = 0.2
LSTM_EPOCHS_PHASE1 = 50
LSTM_EPOCHS_PHASE2 = 100
LSTM_BATCH_SIZE    = 1024
LSTM_LR_PHASE1     = 0.001
LSTM_LR_PHASE2     = 0.0005
LSTM_PATIENCE_P1   = 10
LSTM_PATIENCE_P2   = 15

print('Configuration loaded.')

---
## 3. Feature Engineering Pipeline

The `process_dataset()` function encapsulates the **entire** feature engineering
pipeline for any C-MAPSS dataset:
- Load raw train/test/RUL files
- Construct capped RUL labels
- **Auto-detect** and drop low-variance columns (handles the op-settings difference between FD001/FD003 vs FD002/FD004)
- Apply rolling-window mean/std statistics
- MinMaxScaler normalization

In [ ]:
def process_dataset(ds_id):
    """Full feature engineering pipeline for a single C-MAPSS dataset.
    
    Returns:
        train_df   – processed training DataFrame
        test_df    – processed test DataFrame
        feature_cols – list of feature column names after processing
        y_test     – ground-truth RUL array for test engines
    """
    print(f'\n{"="*60}')
    print(f'  Processing {ds_id}')
    print(f'{"="*60}')
    
    # ── 1. Load data ────────────────────────────────────────────
    train_df = pd.read_csv(f'{DATA}/train_{ds_id}.txt',
                           sep=r'\s+', header=None, names=COLS)
    test_df  = pd.read_csv(f'{DATA}/test_{ds_id}.txt',
                           sep=r'\s+', header=None, names=COLS)
    rul_df   = pd.read_csv(f'{DATA}/RUL_{ds_id}.txt',
                           sep=r'\s+', header=None, names=['RUL'])
    y_test   = rul_df['RUL'].values
    
    print(f'  Train shape: {train_df.shape}')
    print(f'  Test shape:  {test_df.shape}')
    print(f'  Test engines: {len(y_test)}')
    
    # ── 2. RUL labels (capped) ──────────────────────────────────
    ru_max = train_df.groupby('unit')['cycle'].max().reset_index()
    ru_max.columns = ['unit', 'max_cycle']
    train_df = pd.merge(train_df, ru_max, on='unit')
    train_df['RUL'] = (train_df['max_cycle'] - train_df['cycle']).clip(upper=RUL_CAP)
    train_df.drop('max_cycle', axis=1, inplace=True)
    
    # ── 3. Auto-detect low-variance columns ─────────────────────
    sensor_and_op_cols = [c for c in train_df.columns
                          if c.startswith('s') or c.startswith('op')]
    stds = train_df[sensor_and_op_cols].std()
    drop_cols = stds[stds < VARIANCE_THRESH].index.tolist()
    print(f'  Dropping low-variance columns ({len(drop_cols)}): {drop_cols}')
    
    train_df.drop(columns=drop_cols, inplace=True)
    test_df.drop(columns=drop_cols, inplace=True)
    
    # ── 4. Rolling-window features ──────────────────────────────
    remaining = [c for c in train_df.columns
                 if c.startswith('s') or c.startswith('op')]
    
    def _rolling(df):
        rm = (df.groupby('unit')[remaining]
                .rolling(ROLLING_WINDOW, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True))
        rs = (df.groupby('unit')[remaining]
                .rolling(ROLLING_WINDOW, min_periods=1)
                .std()
                .reset_index(level=0, drop=True)
                .fillna(0))
        rm.columns = [f'{c}_avg_{ROLLING_WINDOW}' for c in rm.columns]
        rs.columns = [f'{c}_std_{ROLLING_WINDOW}' for c in rs.columns]
        return pd.concat([df, rm, rs], axis=1)
    
    train_df = _rolling(train_df)
    test_df  = _rolling(test_df)
    
    # ── 5. Normalize ────────────────────────────────────────────
    exclude = ['unit', 'cycle', 'RUL']
    feature_cols = [c for c in train_df.columns if c not in exclude]
    
    scaler = MinMaxScaler()
    train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    test_df[feature_cols]  = scaler.transform(test_df[feature_cols])
    
    print(f'  Final feature count: {len(feature_cols)}')
    print(f'  Train rows: {len(train_df):,}  |  Test rows: {len(test_df):,}')
    
    return train_df, test_df, feature_cols, y_test

print('process_dataset() defined.')

---
## 4. Model Definitions

### 4a. XGBoost Regressor

In [ ]:
def train_xgboost(train_df, test_df, feature_cols, y_test, ds_id):
    """Train XGBoost with GridSearchCV and return metrics."""
    print(f'\n── XGBoost [{ds_id}] ──')
    t0 = time.time()
    
    X_train = train_df[feature_cols].values
    y_train = train_df['RUL'].values
    
    # Extract last cycle per test engine
    X_test = test_df.groupby('unit')[feature_cols].last().values
    
    # Hyperparameter grid (balanced for speed)
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [5, 7],
        'learning_rate': [0.05, 0.1],
        'subsample': [0.8]
    }
    
    xgb = XGBRegressor(random_state=42, n_jobs=-1)
    grid = GridSearchCV(xgb, param_grid, cv=3, scoring='neg_mean_squared_error',
                        n_jobs=-1, verbose=0)
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    preds = best_model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    ns   = nasa_score(y_test, preds)
    
    elapsed = time.time() - t0
    print(f'  Best params: {grid.best_params_}')
    print(f'  RMSE: {rmse:.2f}  |  MAE: {mae:.2f}  |  NASA: {ns:.2f}  ({elapsed:.1f}s)')
    
    return {'Dataset': ds_id, 'Model': 'XGBoost',
            'RMSE': round(rmse, 2), 'MAE': round(mae, 2),
            'NASA_Score': round(ns, 2)}

print('train_xgboost() defined.')

### 4b. SVR (Support Vector Regression)

In [ ]:
def train_svr(train_df, test_df, feature_cols, y_test, ds_id):
    """Train SVR on a sampled subset and return metrics."""
    print(f'\n── SVR [{ds_id}] ──')
    t0 = time.time()
    
    # Sample to keep SVR tractable
    if len(train_df) > SVR_SAMPLE_SIZE:
        sample_df = train_df.sample(n=SVR_SAMPLE_SIZE, random_state=42)
        print(f'  Sampled {SVR_SAMPLE_SIZE:,} rows from {len(train_df):,}')
    else:
        sample_df = train_df
    
    X_train = sample_df[feature_cols].values
    y_train = sample_df['RUL'].values
    X_test  = test_df.groupby('unit')[feature_cols].last().values
    
    # Grid search
    param_grid = {
        'C': [10, 50],
        'kernel': ['rbf'],
        'gamma': ['scale'],
        'epsilon': [1, 5]
    }
    
    svr = SVR()
    grid = GridSearchCV(svr, param_grid, cv=3, scoring='neg_mean_squared_error',
                        n_jobs=-1, verbose=0)
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    preds = best_model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    ns   = nasa_score(y_test, preds)
    
    elapsed = time.time() - t0
    print(f'  Best params: {grid.best_params_}')
    print(f'  RMSE: {rmse:.2f}  |  MAE: {mae:.2f}  |  NASA: {ns:.2f}  ({elapsed:.1f}s)')
    
    return {'Dataset': ds_id, 'Model': 'SVR',
            'RMSE': round(rmse, 2), 'MAE': round(mae, 2),
            'NASA_Score': round(ns, 2)}

print('train_svr() defined.')

### 4c. LSTM (Two-Phase Training)

In [ ]:
# ── LSTM Architecture ───────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=100, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # last timestep
        return self.fc(out)


# ── Sliding-Window Generators ───────────────────────────────────
def make_train_sequences(df, feature_cols, seq_len):
    """Create sliding-window sequences from training data."""
    X_seq, y_seq = [], []
    for unit in df['unit'].unique():
        ud = df[df['unit'] == unit]
        if len(ud) >= seq_len:
            arr = ud[feature_cols].values
            rul = ud['RUL'].values
            for i in range(len(arr) - seq_len):
                X_seq.append(arr[i:i+seq_len])
                y_seq.append(rul[i+seq_len])
    return np.array(X_seq), np.array(y_seq)


def make_test_sequences(df, feature_cols, seq_len):
    """Extract last-window sequence per engine for test evaluation."""
    X_seq = []
    for unit in sorted(df['unit'].unique()):
        ud = df[df['unit'] == unit]
        if len(ud) >= seq_len:
            X_seq.append(ud[feature_cols].values[-seq_len:])
        else:
            pad = seq_len - len(ud)
            X_seq.append(
                np.pad(ud[feature_cols].values,
                       ((pad, 0), (0, 0)), mode='constant'))
    return np.array(X_seq)


# ── Training Function (single phase) ────────────────────────────
def _train_phase(model, train_loader, val_loader, epochs, patience, lr, ds_id, phase):
    """One training phase with early stopping & LR scheduling."""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)
    
    best_val = float('inf')
    counter  = 0
    ckpt     = f'{BASE}/multi_best_{ds_id}_p{phase}.pt'
    
    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)
        
        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                loss = criterion(model(xb), yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f'    P{phase} Epoch {epoch+1}/{epochs} | '
                  f'Train: {train_loss:.2f} | Val: {val_loss:.2f}')
        
        if val_loss < best_val:
            best_val = val_loss
            counter = 0
            torch.save(model.state_dict(), ckpt)
        else:
            counter += 1
        
        if counter >= patience:
            print(f'    P{phase} Early stop at epoch {epoch+1}')
            break
    
    model.load_state_dict(torch.load(ckpt, map_location=device))
    print(f'    P{phase} Best val loss: {best_val:.2f}')
    return model


# ── Full LSTM Pipeline ──────────────────────────────────────────
def train_lstm(train_df, test_df, feature_cols, y_test, ds_id):
    """Two-phase LSTM training and evaluation."""
    print(f'\n── LSTM [{ds_id}] ──')
    t0 = time.time()
    
    # Sequences
    X_train_seq, y_train_seq = make_train_sequences(
        train_df, feature_cols, SEQUENCE_LENGTH)
    X_test_seq = make_test_sequences(
        test_df, feature_cols, SEQUENCE_LENGTH)
    print(f'  Train sequences: {X_train_seq.shape}')
    print(f'  Test sequences:  {X_test_seq.shape}')
    
    # Train/Val split
    Xt, Xv, yt, yv = train_test_split(
        X_train_seq, y_train_seq, test_size=0.2, random_state=42)
    
    Xt_t = torch.tensor(Xt, dtype=torch.float32).to(device)
    yt_t = torch.tensor(yt, dtype=torch.float32).view(-1, 1).to(device)
    Xv_t = torch.tensor(Xv, dtype=torch.float32).to(device)
    yv_t = torch.tensor(yv, dtype=torch.float32).view(-1, 1).to(device)
    
    train_loader = DataLoader(
        TensorDataset(Xt_t, yt_t), batch_size=LSTM_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(
        TensorDataset(Xv_t, yv_t), batch_size=LSTM_BATCH_SIZE, shuffle=False)
    
    input_size = len(feature_cols)
    model = LSTMModel(input_size, LSTM_HIDDEN, LSTM_LAYERS, LSTM_DROPOUT).to(device)
    
    # Phase 1
    print(f'  Phase 1: {LSTM_EPOCHS_PHASE1} epochs, lr={LSTM_LR_PHASE1}')
    model = _train_phase(model, train_loader, val_loader,
                         LSTM_EPOCHS_PHASE1, LSTM_PATIENCE_P1,
                         LSTM_LR_PHASE1, ds_id, phase=1)
    
    # Phase 2 (extended, lower LR)
    print(f'  Phase 2: {LSTM_EPOCHS_PHASE2} epochs, lr={LSTM_LR_PHASE2}')
    model = _train_phase(model, train_loader, val_loader,
                         LSTM_EPOCHS_PHASE2, LSTM_PATIENCE_P2,
                         LSTM_LR_PHASE2, ds_id, phase=2)
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        X_test_t = torch.tensor(X_test_seq, dtype=torch.float32).to(device)
        preds = model(X_test_t).cpu().numpy().flatten()
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    ns   = nasa_score(y_test, preds)
    
    elapsed = time.time() - t0
    print(f'  RMSE: {rmse:.2f}  |  MAE: {mae:.2f}  |  NASA: {ns:.2f}  ({elapsed:.1f}s)')
    
    # Cleanup GPU memory
    del model, Xt_t, yt_t, Xv_t, yv_t, X_test_t
    torch.cuda.empty_cache()
    
    return {'Dataset': ds_id, 'Model': 'LSTM',
            'RMSE': round(rmse, 2), 'MAE': round(mae, 2),
            'NASA_Score': round(ns, 2)}

print('LSTM model & helpers defined.')

---
## 5. Run All Datasets

This is the main execution loop. For each of the 4 datasets:
1. Engineer features via `process_dataset()`
2. Train & evaluate XGBoost, SVR, and LSTM
3. Collect results

In [ ]:
all_results = []
total_start = time.time()

for ds_id in DATASETS:
    # Feature engineering
    train_df, test_df, feature_cols, y_test = process_dataset(ds_id)
    
    # Train & evaluate each model
    all_results.append(train_xgboost(train_df, test_df, feature_cols, y_test, ds_id))
    all_results.append(train_svr(train_df, test_df, feature_cols, y_test, ds_id))
    all_results.append(train_lstm(train_df, test_df, feature_cols, y_test, ds_id))

total_elapsed = time.time() - total_start
print(f'\n{"="*60}')
print(f'  Total runtime: {total_elapsed/60:.1f} minutes')
print(f'{"="*60}')

---
## 6. Cross-Dataset Comparison Table

In [ ]:
results_df = pd.DataFrame(all_results)

# Pivot for a cleaner view
print('\n╔══════════════════════════════════════════════════════════════╗')
print('║         CROSS-DATASET MODEL COMPARISON (Lower = Better)     ║')
print('╚══════════════════════════════════════════════════════════════╝\n')

for metric in ['RMSE', 'MAE', 'NASA_Score']:
    pivot = results_df.pivot(index='Model', columns='Dataset', values=metric)
    pivot = pivot[DATASETS]  # enforce column order
    print(f'\n── {metric} ──')
    display(pivot.style.highlight_min(axis=0, color='#c6efce'))

# Full table
print('\n── Full Results ──')
display(results_df)

---
## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
palette = {'XGBoost': '#2196F3', 'SVR': '#FF9800', 'LSTM': '#4CAF50'}

for ax, metric, title in zip(axes,
                              ['RMSE', 'MAE', 'NASA_Score'],
                              ['RMSE (↓)', 'MAE (↓)', 'NASA Score (↓)']):
    sns.barplot(data=results_df, x='Dataset', y=metric, hue='Model',
                palette=palette, ax=ax, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(metric)
    
    # Add value labels on bars
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f', fontsize=8, padding=2)
    
    ax.legend(title='Model', fontsize=9)

fig.suptitle('Cross-Dataset Model Comparison — NASA C-MAPSS',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{BASE}/multi_dataset_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to Google Drive.')

---
## 8. Per-Dataset Best Model

In [ ]:
print('\n── Best Model per Dataset (by NASA Score) ──\n')

best_per_ds = (results_df
               .loc[results_df.groupby('Dataset')['NASA_Score'].idxmin()]
               .set_index('Dataset'))
display(best_per_ds[['Model', 'RMSE', 'MAE', 'NASA_Score']])

# Difficulty ranking
print('\n── Dataset Difficulty Ranking (avg NASA Score across models) ──\n')
difficulty = (results_df
              .groupby('Dataset')['NASA_Score']
              .mean()
              .sort_values())
for rank, (ds, score) in enumerate(difficulty.items(), 1):
    print(f'  {rank}. {ds}: avg NASA Score = {score:.0f}')

---
## 9. Save Results

In [ ]:
# Save full results table
results_df.to_csv(f'{DATA}/multi_dataset_results.csv', index=False)
print(f'Results saved to {DATA}/multi_dataset_results.csv')

# Display final summary
print('\n══ SUMMARY ══')
print(f'Datasets evaluated:  {len(DATASETS)}')
print(f'Models per dataset:  3 (XGBoost, SVR, LSTM)')
print(f'Total experiments:   {len(results_df)}')
print(f'Total runtime:       {total_elapsed/60:.1f} minutes')
print('\nAll results saved to Google Drive!')

---
## 10. Observations & Takeaways

After running all 4 datasets, compare the results above with these expected patterns:

1. **FD001** (easiest) and **FD003** should have the lowest errors — they have a single operating condition, reducing sensor variability.

2. **FD002** and **FD004** should be harder — 6 operating conditions introduce significant sensor variation that complicates degradation modeling.

3. **FD004** (hardest) combines 6 operating conditions with 2 fault modes — the most complex scenario.

4. **LSTM** is expected to remain the best model across datasets due to its ability to capture temporal degradation dynamics, though its advantage may shrink on the harder datasets.

5. The **operational settings** (`op1`, `op2`, `op3`) are automatically retained for FD002/FD004 by the variance-based feature selection, providing the models with critical context about operating regimes.

Expected difficulty order: **FD001 ≈ FD003 < FD002 < FD004**